In [1]:
import os
import sys
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project Root:", project_root)

Project Root: /home/akash/Projects/Altrodav


In [2]:
from src.data import load_data, split_data

df = load_data()

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (150, 5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [3]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

print("Training :", X_train.shape)
print("Validation:", X_val.shape)
print("Testing :", X_test.shape)

Training : (105, 4)
Validation: (22, 4)
Testing : (23, 4)


In [4]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])

print(pipeline)

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('model', LogisticRegression(random_state=42))])


In [5]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs"],
    "model__max_iter": [100, 200, 500]
}

print(
    "Total Parameter Combinations:",
    len(param_grid["model__C"])
    * len(param_grid["model__solver"])
    * len(param_grid["model__max_iter"])
)

Total Parameter Combinations: 15


In [6]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Hyperparameter Tuning Completed Successfully!")

Hyperparameter Tuning Completed Successfully!


In [7]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(f"{grid_search.best_score_:.4f}")

Best Parameters:
{'model__C': 1, 'model__max_iter': 100, 'model__solver': 'lbfgs'}

Best Cross Validation Accuracy:
0.9810


In [8]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", round(test_accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.9565

Confusion Matrix:
[[7 0 0]
 [0 8 0]
 [0 1 7]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       0.89      1.00      0.94         8
           2       1.00      0.88      0.93         8

    accuracy                           0.96        23
   macro avg       0.96      0.96      0.96        23
weighted avg       0.96      0.96      0.96        23



In [9]:
comparison = pd.DataFrame({
    "Model": [
        "Default Logistic Regression",
        "Tuned Logistic Regression"
    ],
    "Accuracy": [
        0.9565,          # Task 8 accuracy
        test_accuracy
    ]
})

comparison

,Model,Accuracy
0,Default Logistic Regression,0.956500
1,Tuned Logistic Regression,0.956522


In [10]:
PROJECT_ROOT = os.path.abspath("..")

LOG_DIR = os.path.join(PROJECT_ROOT, "logs")
os.makedirs(LOG_DIR, exist_ok=True)

comparison.to_csv(
    os.path.join(LOG_DIR, "task09_hyperparameter_results.csv"),
    index=False
)

cv_results = pd.DataFrame(grid_search.cv_results_)

cv_results.to_csv(
    os.path.join(LOG_DIR, "task09_cv_results.csv"),
    index=False
)

print("Results saved successfully!")
print("Location:", LOG_DIR)

Results saved successfully!
Location: /home/akash/Projects/Altrodav/logs


In [11]:
print("=" * 50)
print("TASK 09 SUMMARY")
print("=" * 50)

print(f"Best Parameters      : {grid_search.best_params_}")
print(f"Best CV Accuracy     : {grid_search.best_score_:.4f}")
print(f"Test Accuracy        : {test_accuracy:.4f}")

print("\nFiles Saved:")

print(os.path.join(LOG_DIR, "task09_hyperparameter_results.csv"))
print(os.path.join(LOG_DIR, "task09_cv_results.csv"))

print("\nTask 09 Completed Successfully!")

TASK 09 SUMMARY
Best Parameters      : {'model__C': 1, 'model__max_iter': 100, 'model__solver': 'lbfgs'}
Best CV Accuracy     : 0.9810
Test Accuracy        : 0.9565

Files Saved:
/home/akash/Projects/Altrodav/logs/task09_hyperparameter_results.csv
/home/akash/Projects/Altrodav/logs/task09_cv_results.csv

Task 09 Completed Successfully!
